In [ ]:
from google.colab import files

uploaded = files.upload()

Saving traditional NBA stats 25-26.xlsx to traditional NBA stats 25-26.xlsx


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel('traditional NBA stats 25-26.xlsx')
print(df)
raw = pd.read_excel("/content/traditional NBA stats 25-26.xlsx")

print("=" * 65)
print("STEP 1 — Raw Data Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
print()
print(raw.head(3).to_string())
print()

    Unnamed: 0                    Player Team  Age  GP   W   L     Min   PTS  \
0            1              Tyrese Maxey  PHI   25  58  32  26  2232.9  1690   
1            2   Shai Gilgeous-Alexander  OKC   27  51  40  11  1697.0  1621   
2            3          Donovan Mitchell  CLE   29  55  35  20  1840.1  1570   
3            4              Jaylen Brown  BOS   29  54  35  19  1850.3  1568   
4            5               Luka Dončić  LAL   27  48  30  18  1698.1  1559   
5            6           Anthony Edwards  MIN   24  51  32  19  1817.4  1502   
6            7             Jalen Brunson  NYK   29  56  38  18  1936.7  1494   
7            8              Kevin Durant  HOU   37  56  34  22  2048.0  1467   
8            9              Jamal Murray  DEN   29  56  33  23  1967.9  1418   
9           10           Cade Cunningham  DET   24  53  40  13  1855.6  1350   
10          11             Julius Randle  MIN   31  61  38  23  2036.5  1313   
11          12              Nikola Jokić

upload and read data. load the raw dataset and displays its size, column names, and first few rows to understand data before tidying it.

In [ ]:
df = raw.rename(columns={
    "Unnamed: 0": "rank",
    "GP":  "games_played",
    "W":   "wins",
    "L":   "losses",
    "Min": "minutes",
    "PTS": "points",
    "FGM": "fg_made",
    "FGA": "fg_attempted",
    "FG%": "fg_pct",
    "3PM": "three_made",
    "3PA": "three_attempted",
    "3P%": "three_pct",
    "FTM": "ft_made",
    "FTA": "ft_attempted",
    "FT%": "ft_pct",
    "OREB":"off_reb",
    "DREB":"def_reb",
    "REB": "total_reb",
    "AST": "assists",
    "TOV": "turnovers",
    "STL": "steals",
    "BLK": "blocks",
    "PF":  "personal_fouls",
    "FP":  "fantasy_points",
    "DD2": "double_doubles",
    "TD3": "triple_doubles",
    "+/-": "plus_minus"
})

print("STEP 2 — Columns renamed.")
print(df.columns.tolist())
print()

STEP 2 — Columns renamed.
['rank', 'Player', 'Team', 'Age', 'games_played', 'wins', 'losses', 'minutes', 'points', 'fg_made', 'fg_attempted', 'fg_pct', 'three_made', 'three_attempted', 'three_pct', 'ft_made', 'ft_attempted', 'ft_pct', 'off_reb', 'def_reb', 'total_reb', 'assists', 'turnovers', 'steals', 'blocks', 'personal_fouls', 'fantasy_points', 'double_doubles', 'triple_doubles', 'plus_minus']



clean up column names

In [ ]:
stats_to_compare = ['points', 'assists', 'total_reb', 'steals', 'blocks']
df_tidy = pd.melt(
    df,
    id_vars=['Player', 'Team', 'Age', 'games_played'],
    value_vars=stats_to_compare,
    var_name='stat_type',
    value_name='total_count'
)
print(df_tidy)

                      Player Team  Age  games_played stat_type  total_count
0               Tyrese Maxey  PHI   25            58    points         1690
1    Shai Gilgeous-Alexander  OKC   27            51    points         1621
2           Donovan Mitchell  CLE   29            55    points         1570
3               Jaylen Brown  BOS   29            54    points         1568
4                Luka Dončić  LAL   27            48    points         1559
..                       ...  ...  ...           ...       ...          ...
245             Ryan Rollins  MIL   23            57    blocks           24
246          Zion Williamson  NOP   25            45    blocks           27
247            Mikal Bridges  NYK   29            61    blocks           49
248             De'Aaron Fox  SAS   28            51    blocks           13
249               Saddiq Bey  NOP   26            55    blocks            3

[250 rows x 6 columns]


choose stats to compare and use .melt() to go from wide to long

In [ ]:
df_final = (
    df_tidy
    .assign(per_game = lambda x: x['total_count'] / x['games_played'])
    .sort_values(['Player', 'stat_type'])
)
print(df_final.head(10))

             Player Team  Age  games_played  stat_type  total_count   per_game
84   Alperen Sengun  HOU   23            52    assists          330   6.346154
234  Alperen Sengun  HOU   23            52     blocks           57   1.096154
34   Alperen Sengun  HOU   23            52     points         1052  20.230769
184  Alperen Sengun  HOU   23            52     steals           70   1.346154
134  Alperen Sengun  HOU   23            52  total_reb          473   9.096154
92    Amen Thompson  HOU   23            57    assists          300   5.263158
242   Amen Thompson  HOU   23            57     blocks           36   0.631579
42    Amen Thompson  HOU   23            57     points          987  17.315789
192   Amen Thompson  HOU   23            57     steals           85   1.491228
142   Amen Thompson  HOU   23            57  total_reb          434   7.614035


use lamba() to calculate per-game stats for each player. Sort data, print.

In [ ]:
df_wide = df_final.pivot_table(
    index=['Player', 'Team', 'Age', 'games_played'],
    columns='stat_type',
    values='per_game'
).reset_index()

df_wide.rename(columns={
    'points': 'ppg',
    'assists': 'apg',
    'total_reb': 'rpg',
    'steals': 'spg',
    'blocks': 'bpg'
}, inplace=True)

df_wide = df_wide[['Player', 'Team', 'Age', 'games_played', 'ppg', 'apg', 'rpg', 'spg', 'bpg']]

print("DataFrame in wide format (first 10 rows):")
print(df_wide.head(10).to_string())

DataFrame in wide format (first 10 rows):
stat_type           Player Team  Age  games_played        ppg       apg       rpg       spg       bpg
0           Alperen Sengun  HOU   23            52  20.230769  6.346154  9.096154  1.346154  1.096154
1            Amen Thompson  HOU   23            57  17.315789  5.263158  7.614035  1.491228  0.631579
2          Anthony Edwards  MIN   24            51  29.450980  3.725490  5.196078  1.352941  0.823529
3              Bam Adebayo  MIA   28            53  18.603774  2.830189  9.867925  1.000000  0.660377
4           Brandon Ingram  TOR   28            58  21.862069  3.810345  5.793103  0.758621  0.775862
5              CJ McCollum  ATL   34            57  18.736842  3.543860  3.421053  0.754386  0.385965
6          Cade Cunningham  DET   24            53  25.471698  9.792453  5.811321  1.509434  0.943396
7             Cooper Flagg  DAL   19            49  20.387755  4.122449  6.551020  1.183673  0.795918
8             De'Aaron Fox  SAS   28    

convert data into wide table and turn stats into columns. calculate per-game stats, reorder columns, and print.